# 2. Train the small AR and NCM generators

Enable a GPU and attach the existing TinyStories preparation output. Set
`DATASET_ROOT` to the folder containing `prepared/`, `codes/`, and `artifacts/`.

Both generators use 6 layers, width 384, and 6 attention heads. The global batch
is 64, with a microbatch of 2 and 100 warmup updates. The first session stops
at update 100 per model so you can measure runtime. The learning-rate schedule
has a 5,000-update horizon. Select one model with `TRAIN_MODELS` if needed.

This notebook saves checkpoints. Use notebook 3 for evaluation.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/kuchidareo/3rd-contribution.git'  # Set the 3rd-contribution repository URL.
REPO_REF = 'main'  # Prefer a full commit containing the baseline implementation.
CHECKOUT = Path('/kaggle/working/3rd-contribution')
REPO = CHECKOUT / '261119-NCM-replicating/ncm'

def run(command, cwd=REPO):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if not CHECKOUT.exists():
    run(['git', 'clone', REPO_URL, CHECKOUT], cwd='/kaggle/working')
run(['git', 'fetch', 'origin', REPO_REF], cwd=CHECKOUT)
run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=CHECKOUT)
required = [REPO / 'baseline_data/prepare.py', REPO / 'training/train_ar.py',
            REPO / 'config/baseline/tinystories.yaml', REPO / 'training/profiles.py']
if any(not p.is_file() for p in required):
    raise RuntimeError('This revision lacks the baseline code. Commit/push the local changes, then set REPO_REF.')
os.chdir(REPO)
# Keep all downloads and environment records inside the preparation/training output.
run(['bash', 'scripts/kaggle_setup.sh'])


In [ ]:
# Fail before any dataset download if the model/training regression tests fail.
run([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
import json
import torch
from omegaconf import OmegaConf

# EDIT: folder containing prepared/, codes/ and artifacts/ in your attached Dataset.
DATASET_ROOT = Path('/kaggle/input/notebooks/kuchidareo/ncm-tiny-stories-kaggle-1st-preparation/ncm-tinystories')
WORK_BASE = Path('/kaggle/working/ncm-experiment')

TRAIN_MODELS = ['ar', 'ncm']  # Set ['ncm'] to train ONLY NCM; ['ar'] for AR only.
PROFILE = 'small'
PRECISION = 'fp16'  # T4: fp16. Native BF16-capable GPU: use bf16 to match paper precision.
MICRO_BATCH = 2
VALIDATE_EVERY = 100  # Use 200 if validation overhead is too high.
CHECKPOINT_EVERY = 100
EARLY_STOPPING = True
EARLY_STOP_PATIENCE = 5  # Consecutive validation checks without sufficient improvement.
EARLY_STOP_MIN_DELTA = 0.001  # Absolute validation-loss improvement.
EARLY_STOP_MIN_STEPS = 100  # Never stops before the configured warmup ends.
# Optional read-only checkpoints from a previous saved Kaggle output.
AR_RESUME = None  # e.g. '/kaggle/input/previous-run/ncm-experiment/small-fp16-selective/runs/ar/latest.pt'
NCM_RESUME = None
# To end this session at an absolute update while keeping the planned LR horizon:
STOP_AT_UPDATE = 100
REBUILD_CODES = False  # True only if the attached code cache is absent/incompatible.

if not TRAIN_MODELS or len(set(TRAIN_MODELS)) != len(TRAIN_MODELS) or set(TRAIN_MODELS) - {'ar', 'ncm'}:
    raise ValueError('TRAIN_MODELS must contain ar and/or ncm, without duplicates.')

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before selecting a training profile.')
if not (DATASET_ROOT / 'prepared/manifest.json').is_file():
    raise FileNotFoundError('Set DATASET_ROOT to the attached folder containing prepared/manifest.json.')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# Reload after Git checkout: notebook kernels otherwise keep old imported functions.
import importlib
import inspect
importlib.invalidate_caches()
import training.profiles as profile_module
if Path(profile_module.__file__).resolve() != (REPO / 'training/profiles.py').resolve():
    raise RuntimeError('training.profiles was imported from another checkout. Restart the kernel and rerun setup.')
profile_module = importlib.reload(profile_module)
configure_profile = profile_module.configure_profile
check_checkpoint_progress = profile_module.check_checkpoint_progress
print('Profile module:', profile_module.__file__)
print('Profile signature:', inspect.signature(configure_profile))
if 'precision' not in inspect.signature(configure_profile).parameters:
    raise RuntimeError('The checked-out training/profiles.py predates FP16 support. Push the updated repository and rerun the clone/checkout cell.')

cfg = OmegaConf.load(REPO / 'config/baseline/tinystories.yaml')
official = OmegaConf.load(REPO / cfg.official_config)
cfg = OmegaConf.create(configure_profile(
    OmegaConf.to_container(cfg, resolve=True),
    OmegaConf.to_container(official, resolve=True),
    PROFILE,
    bf16_supported=(torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported()),
    micro_batch=MICRO_BATCH,
    precision=PRECISION,
))
cfg.training.validation_every = VALIDATE_EVERY
cfg.training.checkpoint_every = CHECKPOINT_EVERY
cfg.training.early_stopping = dict(enabled=EARLY_STOPPING, patience=EARLY_STOP_PATIENCE,
                                  min_delta=EARLY_STOP_MIN_DELTA, min_steps=EARLY_STOP_MIN_STEPS)
RUN_FOLDER = f'{PROFILE}-{cfg.precision}'
if cfg.get('ncm_selective_fp32', False):
    RUN_FOLDER += '-selective'  # Keep the changed NCM numerical policy separate.
WORK = WORK_BASE / RUN_FOLDER
cfg.paths.artifacts = str(DATASET_ROOT / 'artifacts/tinystories')
cfg.paths.tokenizer_checkpoint = str(DATASET_ROOT / 'artifacts/tinystories/tokenizer.pt')
cfg.paths.data = str(DATASET_ROOT / 'prepared')
cfg.paths.codes = str(WORK / 'codes') if REBUILD_CODES else str(DATASET_ROOT / 'codes')
if STOP_AT_UPDATE is not None and not 1 <= STOP_AT_UPDATE <= cfg.training.generator.steps:
    raise ValueError('STOP_AT_UPDATE must be between 1 and the planned total update count.')
WORK.mkdir(parents=True, exist_ok=True)
CONFIG = WORK / 'experiment.yaml'
OmegaConf.save(cfg, CONFIG)

print('Profile:', PROFILE)
print('GPU:', torch.cuda.get_device_name())
print('Output:', WORK)
print('Updates per generator:', cfg.training.generator.steps)
print('Global batch:', cfg.training.generator.global_batch_size)
print('Gradient accumulation:', cfg.experiment.gradient_accumulation_steps)
print('Planned text positions per generator:', cfg.experiment.target_generator_text_positions)
print('Quality samples:', len(cfg.evaluation.seeds), 'seeds x', cfg.evaluation.samples_per_seed)
print('Generation batch:', cfg.evaluation.batch_size)
print('Precision:', cfg.precision, '| compile training:', cfg.training.compile)
print('NCM selective FP32:', cfg.ncm_selective_fp32)
print('NCM policy:', cfg.experiment.ncm_precision_policy)
print('Reconstructed training choices (NOT confirmed author settings):')
print(OmegaConf.to_yaml(cfg.experiment.training_assumptions))
print('Hardware deviations:', list(cfg.experiment.hardware_deviations))
print('Validation/checkpoint intervals:', VALIDATE_EVERY, CHECKPOINT_EVERY)
print('Early stopping:', OmegaConf.to_yaml(cfg.training.early_stopping))
print('Session stop:', STOP_AT_UPDATE)
print('AR architecture:', {k: cfg.ar[k] for k in ('n_layer', 'n_embd', 'n_head')})
print('NCM architecture:', OmegaConf.to_container(cfg.upstream_overrides.model))


In [ ]:
# Read-only cache reuse: no dataset download or copy is needed.
needs_codes = 'ncm' in TRAIN_MODELS
if needs_codes and REBUILD_CODES:
    if not (Path(cfg.paths.codes) / 'manifest.json').exists():
        run([sys.executable, '-m', 'baseline_data.encode_codes', '--config', CONFIG,
             '--batch-size', '16'])
elif needs_codes and not (Path(cfg.paths.codes) / 'manifest.json').is_file():
    raise FileNotFoundError('Attach cached codes or set REBUILD_CODES=True and rerun the configuration cell.')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from training.runtime import load_config
from training.train import verify_data
verify_data(load_config(CONFIG), with_codes=needs_codes)
print('Attached dataset and required assets verified.')


In [ ]:
def train_model(kind, resume=None):
    out = WORK / 'runs' / kind
    command = [sys.executable, '-m', f'training.train_{kind}', '--config', CONFIG, '--out', out]
    checkpoint = out / 'latest.pt'
    source = checkpoint if checkpoint.exists() else (Path(resume) if resume else None)
    completed = 0
    if source is not None:
        # CPU mmap reads metadata without bringing optimizer tensors onto the GPU.
        header = torch.load(source, map_location='cpu', weights_only=False, mmap=True)
        completed = check_checkpoint_progress(kind, header, PROFILE, cfg.training.generator.steps, precision=cfg.precision)
        del header
        command += ['--resume', source]
    stop = STOP_AT_UPDATE if STOP_AT_UPDATE is not None else cfg.training.generator.steps
    if stop < completed:
        raise ValueError(f'{kind}: checkpoint already has {completed} updates; session stop is {stop}.')
    if STOP_AT_UPDATE is not None:
        command += ['--max-steps', str(STOP_AT_UPDATE)]
    print(f'{kind}: completed {completed}; session target {stop}; full target {cfg.training.generator.steps}')
    run(command)

if 'ar' in TRAIN_MODELS:
    train_model('ar', AR_RESUME)
else:
    print('AR training skipped by TRAIN_MODELS.')


In [ ]:
if 'ncm' in TRAIN_MODELS:
    train_model('ncm', NCM_RESUME)
else:
    print('NCM training skipped by TRAIN_MODELS.')


## Save, resume, and evaluate

Save the first session's outputs after both models reach update 100. Check runtime
and validation loss before setting a higher absolute `STOP_AT_UPDATE`. The 5,000
updates are a learning-rate horizon, with no guarantee of fitting in one session.

Preserve `experiment.yaml` and both `latest.pt` and `best.pt` for each model.
On a new session, attach the saved output and point `AR_RESUME` or `NCM_RESUME`
to the corresponding `latest.pt`. Start the small models with fresh weights.
The default output folder is `ncm-experiment/small-fp16-selective/`.

For a comparison at update 100, give notebook 3 the `latest.pt` from each model
at that update. Independently selected `best.pt` files can have different exposures.
